I'll to replicate what the functions `make_ts_events`, `make_ts_power` and `aggregate_ts` from `utils.py` do, but for the ERA5 data.

In [3]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, Normalize
import seaborn as sns
from statsmodels.tsa.stattools import grangercausalitytests, ccf
import numpy as np
import plotly.express as px
from datetime import datetime
import geopandas as gpd
from shapely.geometry import Point
import warnings
import xarray as xr
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
from utils import *

We load the data from a `.grib` file.

In [4]:
grib_file_path = '../Data/ERA5_land/reanalysis-era5-land-2014.grib'
ds = xr.open_dataset(grib_file_path, engine='cfgrib')

In [5]:
ds

<xarray.Dataset> Size: 5GB
Dimensions:     (time: 366, step: 4, latitude: 261, longitude: 591)
Coordinates:
    number      int32 4B ...
  * time        (time) datetime64[ns] 3kB 2013-12-31 2014-01-01 ... 2014-12-31
  * step        (step) timedelta64[ns] 32B 06:00:00 12:00:00 ... 1 days 00:00:00
    surface     float64 8B ...
  * latitude    (latitude) float64 2kB 50.0 49.9 49.8 49.7 ... 24.2 24.1 24.0
  * longitude   (longitude) float64 5kB -125.0 -124.9 -124.8 ... -66.1 -66.0
    valid_time  (time, step) datetime64[ns] 12kB ...
Data variables:
    t2m         (time, step, latitude, longitude) float32 903MB ...
    u10         (time, step, latitude, longitude) float32 903MB ...
    v10         (time, step, latitude, longitude) float32 903MB ...
    sp          (time, step, latitude, longitude) float32 903MB ...
    tp          (time, step, latitude, longitude) float32 903MB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-03-11T19:40 GRIB to CDM+CF via cfgrib-0.9.1...

In [44]:
num_samples = 5
random_indices = np.random.choice(ds.latitude.size, num_samples, replace=False)
sample = ds.isel(latitude=random_indices)
random_indices = np.random.choice(sample.time.size, num_samples, replace=False)
sample = sample.isel(time=random_indices)
print(sample)

<xarray.Dataset> Size: 1MB
Dimensions:     (time: 5, step: 4, latitude: 5, longitude: 591)
Coordinates:
    number      int32 4B 0
  * time        (time) datetime64[ns] 40B 2014-09-30 2014-05-03 ... 2014-11-27
  * step        (step) timedelta64[ns] 32B 06:00:00 12:00:00 ... 1 days 00:00:00
    surface     float64 8B 0.0
  * latitude    (latitude) float64 40B 49.0 44.9 42.9 45.7 34.4
  * longitude   (longitude) float64 5kB -125.0 -124.9 -124.8 ... -66.1 -66.0
    valid_time  (time, step) datetime64[ns] 160B ...
Data variables:
    t2m         (time, step, latitude, longitude) float32 236kB ...
    u10         (time, step, latitude, longitude) float32 236kB ...
    v10         (time, step, latitude, longitude) float32 236kB ...
    sp          (time, step, latitude, longitude) float32 236kB ...
    tp          (time, step, latitude, longitude) float32 236kB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Me

In [35]:

# Read the shapefile
shapefile_path = '../Data/cb_2018_us_county_500k/cb_2018_us_county_500k.shp'
gdf = gpd.read_file(shapefile_path)

def get_FIPS(latitude, longitude, gdf):
    #print(f"lat: {latitude.shape} | lon: {longitude.shape}")
    point = Point(longitude, latitude)
    for _, row in gdf.iterrows():
        if row['geometry'].contains(point):
            statefp = row['STATEFP']
            countyfp = row['COUNTYFP']
            return int(statefp+countyfp)
    return None

vec_get_FIPS = np.vectorize(lambda x,y: get_FIPS(x,y,gdf))
# Example usage
latitude = 36.16
longitude = -86.78
info = vec_get_FIPS(latitude, longitude)
print(info)
#sample['FIPS']=xr.DataArray(vec_get_FIPS(sample['latitude'], sample['longitude'], gdf), dims=['latitude', 'longitude'])
#sample

47037


In [45]:
sample.assign(lat_var =(['latitude'],sample['latitude'].values),lon_var =(['longitude'],sample['longitude'].values))

<xarray.Dataset> Size: 1MB
Dimensions:     (time: 5, step: 4, latitude: 5, longitude: 591)
Coordinates:
    number      int32 4B 0
  * time        (time) datetime64[ns] 40B 2014-09-30 2014-05-03 ... 2014-11-27
  * step        (step) timedelta64[ns] 32B 06:00:00 12:00:00 ... 1 days 00:00:00
    surface     float64 8B 0.0
  * latitude    (latitude) float64 40B 49.0 44.9 42.9 45.7 34.4
  * longitude   (longitude) float64 5kB -125.0 -124.9 -124.8 ... -66.1 -66.0
    valid_time  (time, step) datetime64[ns] 160B ...
Data variables:
    t2m         (time, step, latitude, longitude) float32 236kB ...
    u10         (time, step, latitude, longitude) float32 236kB ...
    v10         (time, step, latitude, longitude) float32 236kB ...
    sp          (time, step, latitude, longitude) float32 236kB ...
    tp          (time, step, latitude, longitude) float32 236kB ...
    lat_var     (latitude) float64 40B 49.0 44.9 42.9 45.7 34.4
    lon_var     (longitude) float64 5kB -125.0 -124.9 -124.8 ... -66.1 -66.0
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-03-11T19:40 GRIB to CDM+CF via cfgrib-0.9.1...

In [39]:
fips = xr.apply_ufunc(
    lambda x,y: get_FIPS(x,y,gdf),
    sample['latitude'],
    sample['longitude'],
    vectorize=True
)

In [46]:
sample = sample.assign_coords(FIPS=fips)

In [50]:
sample.dims

FrozenMappingWarningOnValuesAccess({'time': 5, 'step': 4, 'latitude': 5, 'longitude': 591})

In [49]:
sample.dropna(dim='FIPS')

ValueError: Dimension 'FIPS' not found in data dimensions ('time', 'step', 'latitude', 'longitude')